#### Importing libraries

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import os
import operator
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from numpy import unique
from keras.models import Sequential
from keras.layers import LSTM
from keras.layers import Conv1D, Conv2D, Dense, BatchNormalization, Flatten, MaxPooling1D
from keras.layers import Dense, Dropout

In [3]:
pip install np_utils

Note: you may need to restart the kernel to use updated packages.


#### Set path, import datasets

In [5]:
#set file path for data
path = r'/Users/artoe/Documents/DataAnalytics/Machine Learning With Python/Achievement 2/data'

In [6]:
#load unscaled weather data
weather = pd.read_csv(os.path.join(path, 'weather-prediction-processed.csv'), index_col = False)

In [7]:
#load pleasant weather answers data
pleasant = pd.read_csv(os.path.join(path, 'Pleasant_Weather.csv'), index_col = False)

In [8]:
#set tf random seed to ensure reproducible results
tf.random.set_seed(42)

#### Cleaning and preparing data for deep learning

In [10]:
weather.head()

,DATE,MONTH,BASEL_cloud_cover,BASEL_wind_speed,BASEL_humidity,BASEL_pressure,BASEL_global_radiation,BASEL_precipitation,BASEL_snow_depth,BASEL_sunshine,...,VALENTIA_cloud_cover,VALENTIA_humidity,VALENTIA_pressure,VALENTIA_global_radiation,VALENTIA_precipitation,VALENTIA_snow_depth,VALENTIA_sunshine,VALENTIA_temp_mean,VALENTIA_temp_min,VALENTIA_temp_max
0,19600101,1,7,2.1,0.85,1.018,0.32,0.09,0,0.7,...,5,0.88,1.0003,0.45,0.34,0,4.7,8.5,6.0,10.9
1,19600102,1,6,2.1,0.84,1.018,0.36,1.05,0,1.1,...,7,0.91,1.0007,0.25,0.84,0,0.7,8.9,5.6,12.1
2,19600103,1,8,2.1,0.90,1.018,0.18,0.30,0,0.0,...,7,0.91,1.0096,0.17,0.08,0,0.1,10.5,8.1,12.9
3,19600104,1,3,2.1,0.92,1.018,0.58,0.00,0,4.1,...,7,0.86,1.0184,0.13,0.98,0,0.0,7.4,7.3,10.6
4,19600105,1,6,2.1,0.95,1.018,0.65,0.14,0,5.4,...,3,0.80,1.0328,0.46,0.00,0,5.7,5.7,3.0,8.4


In [11]:
pleasant.head()

,DATE,BASEL_pleasant_weather,BELGRADE_pleasant_weather,BUDAPEST_pleasant_weather,DEBILT_pleasant_weather,DUSSELDORF_pleasant_weather,HEATHROW_pleasant_weather,KASSEL_pleasant_weather,LJUBLJANA_pleasant_weather,MAASTRICHT_pleasant_weather,MADRID_pleasant_weather,MUNCHENB_pleasant_weather,OSLO_pleasant_weather,SONNBLICK_pleasant_weather,STOCKHOLM_pleasant_weather,VALENTIA_pleasant_weather
0,19600101,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,19600102,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,19600103,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,19600104,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,19600105,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [12]:
#drop Gdansk, Roma and Tours columns from weather, as these locations are not in the pleasant weather data
drop_cols = [col for col in weather.columns if col.startswith(('GDANSK','ROMA','TOURS'))]
weather.drop(columns=drop_cols, inplace=True)

In [13]:
pleasant.columns

Index(['DATE', 'BASEL_pleasant_weather', 'BELGRADE_pleasant_weather',
       'BUDAPEST_pleasant_weather', 'DEBILT_pleasant_weather',
       'DUSSELDORF_pleasant_weather', 'HEATHROW_pleasant_weather',
       'KASSEL_pleasant_weather', 'LJUBLJANA_pleasant_weather',
       'MAASTRICHT_pleasant_weather', 'MADRID_pleasant_weather',
       'MUNCHENB_pleasant_weather', 'OSLO_pleasant_weather',
       'SONNBLICK_pleasant_weather', 'STOCKHOLM_pleasant_weather',
       'VALENTIA_pleasant_weather'],
      dtype='object')

In [14]:
#find features common to all stations in weather dataset
#start by separating features for each location
basel = collist = weather.filter(like='BASEL').columns.tolist()
belgrade = collist=weather.filter(like='BELGRADE').columns.tolist()
budapest = collist=weather.filter(like='BUDAPEST').columns.tolist()
debilt = collist=weather.filter(like='DEBILT').columns.tolist()
dusseldorf = collist=weather.filter(like='DUSSELDORF').columns.tolist()
heathrow = collist=weather.filter(like='HEATHROW').columns.tolist()
kassel = collist=weather.filter(like='KASSEL').columns.tolist()
ljubljana = collist=weather.filter(like='LJUBLJANA').columns.tolist()
maastricht = collist=weather.filter(like='MAASTRICHT').columns.tolist()
madrid = collist=weather.filter(like='MADRID').columns.tolist()
munchen = collist=weather.filter(like='MUNCHENB').columns.tolist()
oslo = collist=weather.filter(like='OSLO').columns.tolist()
sonnblick = collist=weather.filter(like='SONNBLICK').columns.tolist()
stockholm = collist=weather.filter(like='STOCKHOLM').columns.tolist()
valentia = collist=weather.filter(like='VALENTIA').columns.tolist()

In [15]:
locations = pd.DataFrame([basel, belgrade, budapest, debilt, dusseldorf, heathrow, kassel, ljubljana, maastricht, madrid, munchen, oslo, sonnblick, stockholm, valentia])


In [16]:
locations = locations.T
locations.columns=["basel", "belgrade", "budapest", "debilt", "dusseldorf", "heathrow", "kassel", "ljubljana", "maastricht", "madrid", "munchen", "oslo", "sonnblick", "stockholm", "valentia"]

In [17]:
locations.shape

(11, 15)

In [18]:
locations.head(11)

,basel,belgrade,budapest,debilt,dusseldorf,heathrow,kassel,ljubljana,maastricht,madrid,munchen,oslo,sonnblick,stockholm,valentia
0,BASEL_cloud_cover,BELGRADE_cloud_cover,BUDAPEST_cloud_cover,DEBILT_cloud_cover,DUSSELDORF_cloud_cover,HEATHROW_cloud_cover,KASSEL_wind_speed,LJUBLJANA_cloud_cover,MAASTRICHT_cloud_cover,MADRID_cloud_cover,MUNCHENB_cloud_cover,OSLO_cloud_cover,SONNBLICK_cloud_cover,STOCKHOLM_cloud_cover,VALENTIA_cloud_cover
1,BASEL_wind_speed,BELGRADE_humidity,BUDAPEST_humidity,DEBILT_wind_speed,DUSSELDORF_wind_speed,HEATHROW_humidity,KASSEL_humidity,LJUBLJANA_wind_speed,MAASTRICHT_wind_speed,MADRID_wind_speed,MUNCHENB_humidity,OSLO_wind_speed,SONNBLICK_wind_speed,STOCKHOLM_pressure,VALENTIA_humidity
2,BASEL_humidity,BELGRADE_pressure,BUDAPEST_pressure,DEBILT_humidity,DUSSELDORF_humidity,HEATHROW_pressure,KASSEL_pressure,LJUBLJANA_humidity,MAASTRICHT_humidity,MADRID_humidity,MUNCHENB_global_radiation,OSLO_humidity,SONNBLICK_humidity,STOCKHOLM_global_radiation,VALENTIA_pressure
3,BASEL_pressure,BELGRADE_global_radiation,BUDAPEST_global_radiation,DEBILT_pressure,DUSSELDORF_pressure,HEATHROW_global_radiation,KASSEL_global_radiation,LJUBLJANA_pressure,MAASTRICHT_pressure,MADRID_pressure,MUNCHENB_precipitation,OSLO_pressure,SONNBLICK_pressure,STOCKHOLM_precipitation,VALENTIA_global_radiation
4,BASEL_global_radiation,BELGRADE_precipitation,BUDAPEST_precipitation,DEBILT_global_radiation,DUSSELDORF_global_radiation,HEATHROW_precipitation,KASSEL_precipitation,LJUBLJANA_global_radiation,MAASTRICHT_global_radiation,MADRID_global_radiation,MUNCHENB_snow_depth,OSLO_global_radiation,SONNBLICK_global_radiation,STOCKHOLM_sunshine,VALENTIA_precipitation
5,BASEL_precipitation,BELGRADE_sunshine,BUDAPEST_sunshine,DEBILT_precipitation,DUSSELDORF_precipitation,HEATHROW_snow_depth,KASSEL_sunshine,LJUBLJANA_precipitation,MAASTRICHT_precipitation,MADRID_precipitation,MUNCHENB_sunshine,OSLO_precipitation,SONNBLICK_precipitation,STOCKHOLM_temp_mean,VALENTIA_snow_depth
6,BASEL_snow_depth,BELGRADE_temp_mean,BUDAPEST_temp_mean,DEBILT_sunshine,DUSSELDORF_snow_depth,HEATHROW_sunshine,KASSEL_temp_mean,LJUBLJANA_sunshine,MAASTRICHT_sunshine,MADRID_sunshine,MUNCHENB_temp_mean,OSLO_snow_depth,SONNBLICK_sunshine,STOCKHOLM_temp_min,VALENTIA_sunshine
7,BASEL_sunshine,BELGRADE_temp_min,BUDAPEST_temp_min,DEBILT_temp_mean,DUSSELDORF_sunshine,HEATHROW_temp_mean,KASSEL_temp_min,LJUBLJANA_temp_mean,MAASTRICHT_temp_mean,MADRID_temp_mean,MUNCHENB_temp_min,OSLO_sunshine,SONNBLICK_temp_mean,STOCKHOLM_temp_max,VALENTIA_temp_mean
8,BASEL_temp_mean,BELGRADE_temp_max,BUDAPEST_temp_max,DEBILT_temp_min,DUSSELDORF_temp_mean,HEATHROW_temp_min,KASSEL_temp_max,LJUBLJANA_temp_min,MAASTRICHT_temp_min,MADRID_temp_min,MUNCHENB_temp_max,OSLO_temp_mean,SONNBLICK_temp_min,None,VALENTIA_temp_min
9,BASEL_temp_min,None,None,DEBILT_temp_max,DUSSELDORF_temp_min,HEATHROW_temp_max,None,LJUBLJANA_temp_max,MAASTRICHT_temp_max,MADRID_temp_max,None,OSLO_temp_min,SONNBLICK_temp_max,None,VALENTIA_temp_max


In [19]:
locations.isnull().sum()

basel         0
belgrade      2
budapest      2
debilt        1
dusseldorf    0
heathrow      1
kassel        2
ljubljana     1
maastricht    1
madrid        1
munchen       2
oslo          0
sonnblick     1
stockholm     3
valentia      1
dtype: int64

In [20]:
locations[['basel','dusseldorf','oslo']]

,basel,dusseldorf,oslo
0,BASEL_cloud_cover,DUSSELDORF_cloud_cover,OSLO_cloud_cover
1,BASEL_wind_speed,DUSSELDORF_wind_speed,OSLO_wind_speed
2,BASEL_humidity,DUSSELDORF_humidity,OSLO_humidity
3,BASEL_pressure,DUSSELDORF_pressure,OSLO_pressure
4,BASEL_global_radiation,DUSSELDORF_global_radiation,OSLO_global_radiation
5,BASEL_precipitation,DUSSELDORF_precipitation,OSLO_precipitation
6,BASEL_snow_depth,DUSSELDORF_snow_depth,OSLO_snow_depth
7,BASEL_sunshine,DUSSELDORF_sunshine,OSLO_sunshine
8,BASEL_temp_mean,DUSSELDORF_temp_mean,OSLO_temp_mean
9,BASEL_temp_min,DUSSELDORF_temp_min,OSLO_temp_min


In [21]:
#creating lists of weather observation columns based on these observations.
cloudcover = collist = weather.filter(like='_cloud_cover').columns.tolist()
windspeed = collist = weather.filter(like='_wind_speed').columns.tolist()
humidity = collist = weather.filter(like='_humidity').columns.tolist()
pressure = collist = weather.filter(like='_pressure').columns.tolist()
globalradiation = collist = weather.filter(like='_global_radiation').columns.tolist()
precipitation = collist = weather.filter(like='_precipitation').columns.tolist()
snowdepth = collist = weather.filter(like='_snow_depth').columns.tolist()
sunshine = collist = weather.filter(like='_sunshine').columns.tolist()
tempmean = collist = weather.filter(like='_temp_mean').columns.tolist()
tempmin = collist = weather.filter(like='_temp_min').columns.tolist()
tempmax = collist = weather.filter(like='_temp_max').columns.tolist()


In [22]:
observations = pd.DataFrame([cloudcover, windspeed, humidity, pressure, globalradiation, precipitation, snowdepth, sunshine, tempmean, tempmin, tempmax])
observations = observations.T

In [23]:
observations.columns=['cloudcover', 'windspeed', 'humidity', 'pressure', 'globalradiation', 'precipitation', 'snowdepth', 'sunshine', 'tempmean', 'tempmin', 'tempmax']


In [24]:
observations.isnull().sum()

cloudcover         1
windspeed          6
humidity           1
pressure           1
globalradiation    0
precipitation      0
snowdepth          9
sunshine           0
tempmean           0
tempmin            0
tempmax            0
dtype: int64

In [25]:
#we can drop the columns for windspeed and snow depth, as multiple years are missing
windspeed = [col for col in weather.columns if '_wind_speed' in col]
snowdepth = [col for col in weather.columns if '_snow_depth' in col]

In [26]:
weather.drop(columns=windspeed, inplace=True)
weather.drop(columns=snowdepth, inplace=True)

In [27]:
weather.head()

,DATE,MONTH,BASEL_cloud_cover,BASEL_humidity,BASEL_pressure,BASEL_global_radiation,BASEL_precipitation,BASEL_sunshine,BASEL_temp_mean,BASEL_temp_min,...,STOCKHOLM_temp_max,VALENTIA_cloud_cover,VALENTIA_humidity,VALENTIA_pressure,VALENTIA_global_radiation,VALENTIA_precipitation,VALENTIA_sunshine,VALENTIA_temp_mean,VALENTIA_temp_min,VALENTIA_temp_max
0,19600101,1,7,0.85,1.018,0.32,0.09,0.7,6.5,0.8,...,4.9,5,0.88,1.0003,0.45,0.34,4.7,8.5,6.0,10.9
1,19600102,1,6,0.84,1.018,0.36,1.05,1.1,6.1,3.3,...,5.0,7,0.91,1.0007,0.25,0.84,0.7,8.9,5.6,12.1
2,19600103,1,8,0.90,1.018,0.18,0.30,0.0,8.5,5.1,...,4.1,7,0.91,1.0096,0.17,0.08,0.1,10.5,8.1,12.9
3,19600104,1,3,0.92,1.018,0.58,0.00,4.1,6.3,3.8,...,2.3,7,0.86,1.0184,0.13,0.98,0.0,7.4,7.3,10.6
4,19600105,1,6,0.95,1.018,0.65,0.14,5.4,3.0,-0.7,...,4.3,3,0.80,1.0328,0.46,0.00,5.7,5.7,3.0,8.4


In [28]:
#looking at the remaining missing observations
observations[['cloudcover','humidity','pressure']]

,cloudcover,humidity,pressure
0,BASEL_cloud_cover,BASEL_humidity,BASEL_pressure
1,BELGRADE_cloud_cover,BELGRADE_humidity,BELGRADE_pressure
2,BUDAPEST_cloud_cover,BUDAPEST_humidity,BUDAPEST_pressure
3,DEBILT_cloud_cover,DEBILT_humidity,DEBILT_pressure
4,DUSSELDORF_cloud_cover,DUSSELDORF_humidity,DUSSELDORF_pressure
5,HEATHROW_cloud_cover,HEATHROW_humidity,HEATHROW_pressure
6,LJUBLJANA_cloud_cover,KASSEL_humidity,KASSEL_pressure
7,MAASTRICHT_cloud_cover,LJUBLJANA_humidity,LJUBLJANA_pressure
8,MADRID_cloud_cover,MAASTRICHT_humidity,MAASTRICHT_pressure
9,MUNCHENB_cloud_cover,MADRID_humidity,MADRID_pressure


In [29]:
#we can see that Kassel cloudcover, Munchen pressure and Stockholm humidity are missing.
#we can fill in these missing observations with those from nearby locations.
#Kassel is near Dusseldorf, Munchen is near Sonnblick, Stockholm is near Oslo.

In [30]:
weather['KASSEL_cloud_cover'] = weather['DUSSELDORF_cloud_cover']

In [31]:
weather[['KASSEL_cloud_cover','DUSSELDORF_cloud_cover']]

,KASSEL_cloud_cover,DUSSELDORF_cloud_cover
0,8,8
1,8,8
2,7,7
3,8,8
4,7,7
...,...,...
22945,8,8
22946,7,7
22947,8,8
22948,7,7


In [32]:
#replace the other two missing columns
weather['MUNCHENB_pressure'] = weather['SONNBLICK_pressure']
weather['STOCKHOLM_humidity'] = weather['OSLO_humidity']

In [33]:
weather[['MUNCHENB_pressure', 'SONNBLICK_pressure']]

,MUNCHENB_pressure,SONNBLICK_pressure
0,1.0304,1.0304
1,1.0292,1.0292
2,1.0320,1.0320
3,1.0443,1.0443
4,1.0430,1.0430
...,...,...
22945,1.0263,1.0263
22946,1.0263,1.0263
22947,1.0263,1.0263
22948,1.0263,1.0263


In [34]:
weather[['STOCKHOLM_humidity','OSLO_humidity']]

,STOCKHOLM_humidity,OSLO_humidity
0,0.98,0.98
1,0.62,0.62
2,0.69,0.69
3,0.98,0.98
4,0.96,0.96
...,...,...
22945,0.98,0.98
22946,1.00,1.00
22947,0.85,0.85
22948,0.94,0.94


In [35]:
weather.shape

(22950, 137)

In [36]:
pleasant.shape

(22950, 16)

In [37]:
#export cleaned weather data
weather.to_csv(os.path.join(path, 'weather_cleaned.csv'), index=False)

In [74]:
#drop date/date+month columns
weather.drop(columns=['DATE','MONTH'], inplace=True)

In [76]:
pleasant.drop(columns='DATE', inplace=True)

In [78]:
weather.shape

(22950, 135)

In [80]:
pleasant.shape

(22950, 15)

#### Reshape cleaned data for deep learning

In [41]:
#rename dataframes so that the weather data is X and the pleasant weather answers are y
X = weather
y = pleasant

In [42]:
#convert these to arrays
X = np.array(X)
y = np.array(y)

In [43]:
X

array([[7.    , 0.85  , 1.018 , ..., 8.    , 1.0304, 0.98  ],
       [6.    , 0.84  , 1.018 , ..., 8.    , 1.0292, 0.62  ],
       [8.    , 0.9   , 1.018 , ..., 7.    , 1.032 , 0.69  ],
       ...,
       [4.    , 0.76  , 1.0227, ..., 8.    , 1.0263, 0.85  ],
       [5.    , 0.8   , 1.0212, ..., 7.    , 1.0263, 0.94  ],
       [5.    , 0.84  , 1.0193, ..., 8.    , 1.0263, 0.97  ]])

In [44]:
X.shape

(22950, 135)

In [45]:
y.shape

(22950, 15)

In [46]:
#the X set has 22950 rows with 90 columns. 
#the 135 columns consist of 9 observations (cloud cover, global radiation, humidity, precipitation, pressure, sunshine, mean-, min- and max-temperature) for 15 locations.
X = X.reshape(-1,15,9)

In [47]:
X.shape

(22950, 15, 9)

In [48]:
X

array([[[ 7.0000e+00,  8.5000e-01,  1.0180e+00, ...,  6.5000e+00,
          8.0000e-01,  1.0900e+01],
        [ 1.0000e+00,  8.1000e-01,  1.0195e+00, ...,  3.7000e+00,
         -9.0000e-01,  7.9000e+00],
        [ 4.0000e+00,  6.7000e-01,  1.0170e+00, ...,  2.4000e+00,
         -4.0000e-01,  5.1000e+00],
        ...,
        [ 1.0304e+00,  4.8000e-01,  1.0000e-02, ..., -3.2000e+00,
          5.0000e+00,  1.0114e+00],
        [ 5.0000e-02,  3.2000e-01,  0.0000e+00, ...,  5.0000e+00,
          8.8000e-01,  1.0003e+00],
        [ 4.5000e-01,  3.4000e-01,  4.7000e+00, ...,  8.0000e+00,
          1.0304e+00,  9.8000e-01]],

       [[ 6.0000e+00,  8.4000e-01,  1.0180e+00, ...,  6.1000e+00,
          3.3000e+00,  1.0100e+01],
        [ 6.0000e+00,  8.4000e-01,  1.0172e+00, ...,  2.9000e+00,
          2.2000e+00,  4.4000e+00],
        [ 4.0000e+00,  6.7000e-01,  1.0170e+00, ...,  2.3000e+00,
          1.4000e+00,  3.1000e+00],
        ...,
        [ 1.0292e+00,  2.1000e-01,  6.1000e-01, ..., -

In [49]:
#split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X,y,random_state=42)

In [50]:
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(17212, 15, 9) (17212, 15)
(5738, 15, 9) (5738, 15)


### Create Keras CNN model

#### Initial Parameters

In [53]:
epochs = 30
batch_size = 16
n_hidden = 4

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(Conv1D(n_hidden, kernel_size=2, activation='relu', input_shape=(timesteps, input_dim)))
model.add(Dense(2, activation='relu')) 
model.add(MaxPooling1D())
model.add(Flatten())
model.add(Dense(n_classes, activation='softmax'))

C:\Users\artoe\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [54]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                      │ (None, 14, 4)               │              76 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 14, 2)               │              10 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d (MaxPooling1D)         │ (None, 7, 2)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 14)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 15)                  │             225 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 311 (1.21 KB)

 Trainable params: 311 (1.21 KB)

 Non-trainable params: 0 (0.00 B)

In [55]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [56]:
model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, verbose=1)

Epoch 1/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.1115 - loss: 148.9618
Epoch 2/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.0970 - loss: 1586.0302
Epoch 3/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.0948 - loss: 5284.6558
Epoch 4/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.0948 - loss: 11316.7676
Epoch 5/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.0945 - loss: 19796.6680
Epoch 6/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.0962 - loss: 30911.5605
Epoch 7/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.0975 - loss: 44915.6367
Epoch 8/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.0967 - loss: 61965.7969
Epoch 9/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.0996 - loss: 82323.6797
Epoch 10/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.1001 - loss: 106192.6172
Epoch 11/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.1041 - loss: 

#### Show accuracy of model with confusion matrix

In [58]:
#set location names
locations = {
    0: 'BASEL',
    1: 'BELGRADE',
    2: 'BUDAPEST',
    3: 'DEBILT',
    4: 'DUSSELDORF',
    5: 'HEATHROW',
    6: 'KASSEL',
    7: 'LJUBLJANA',
    8: 'MAASTRICHT',
    9: 'MADRID',
    10: 'MUNCHENB',
    11: 'OSLO',
    12: 'SONNBLICK',
    13: 'STOCKHOLM',
    14: 'VALENTIA',
}

In [59]:
def confusion_matrix(Y_true, Y_pred):
    Y_true = pd.Series([locations[y] for y in np.argmax(Y_true, axis=1)])
    Y_pred = pd.Series([locations[y] for y in np.argmax(Y_pred, axis=1)])

    return pd.crosstab(Y_true, Y_pred, rownames=['True'], colnames=['Pred'])

In [60]:
#evaluate
print(confusion_matrix(y_test, model.predict(X_test)))

180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Pred        BASEL  BELGRADE  BUDAPEST  DEBILT  DUSSELDORF  HEATHROW  KASSEL  \
True                                                                          
BASEL          45       886        39     195         176       132     234   
BELGRADE        3       445         0      12          30         3      13   
BUDAPEST        0        84         2       0           3         0       2   
DEBILT          0        30         1       0           4         0       0   
DUSSELDORF      0         7         0       0           0         0       0   
HEATHROW        0        31         0       1           5         0       1   
KASSEL          1         4         0       0           0         0       0   
LJUBLJANA       2        19         0       2           1         0       2   
MAASTRICHT      0         3         0       2           1         1       0   
MADRID          2       129         2       1          44         4      40   
MUNCHENB   

#### Final Parameters

In [62]:
epochs = 60
batch_size = 9
n_hidden = 64
model = Sequential()
model.add(Conv1D(n_hidden, kernel_size=3, activation='relu', input_shape=(timesteps, input_dim)))
model.add(Dense(32, activation='relu'))
model.add(MaxPooling1D())
model.add(Flatten())
model.add(Dense(n_classes, activation='softmax'))

C:\Users\artoe\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [63]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d_1 (Conv1D)                    │ (None, 13, 64)              │           1,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 13, 32)              │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_1 (MaxPooling1D)       │ (None, 6, 32)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 192)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 15)                  │           2,895 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,767 (26.43 KB)

 Trainable params: 6,767 (26.43 KB)

 Non-trainable params: 0 (0.00 B)

In [64]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [65]:
model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, verbose=1)

Epoch 1/60
1913/1913 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.1092 - loss: 41561.6133
Epoch 2/60
1913/1913 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.1178 - loss: 920138.0625
Epoch 3/60
1913/1913 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.1127 - loss: 3551816.7500
Epoch 4/60
1913/1913 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.1138 - loss: 8276490.5000
Epoch 5/60
1913/1913 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.1191 - loss: 15470406.0000
Epoch 6/60
1913/1913 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.1217 - loss: 25611606.0000
Epoch 7/60
1913/1913 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.1196 - loss: 39081532.0000
Epoch 8/60
1913/1913 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.1268 - loss: 56411792.0000
Epoch 9/60
1913/1913 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.1241 - loss: 77785664.0000
Epoch 10/60
1913/1913 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.1219 - loss: 104008056.0000
Epoch 11/60
1913/1913 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step 

In [66]:
#evaluating model performance with confusion matrix
def confusion_matrix(Y_true, Y_pred):
    Y_true = pd.Series([locations[y] for y in np.argmax(Y_true, axis=1)])
    Y_pred = pd.Series([locations[y] for y in np.argmax(Y_pred, axis=1)])

    return pd.crosstab(Y_true, Y_pred, rownames=['True'], colnames=['Pred'])

In [67]:
print(confusion_matrix(y_test, model.predict(X_test)))

180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Pred        BASEL  BELGRADE  BUDAPEST  DEBILT  DUSSELDORF  HEATHROW  KASSEL  \
True                                                                          
BASEL           5        60       269     100         136         4       8   
BELGRADE        0        23       122       0           1         0       0   
BUDAPEST        0         2        16       0           0         0       0   
DEBILT          0         0         3       0           1         0       0   
DUSSELDORF      0         0         0       0           0         0       0   
HEATHROW        0         0         1       0           3         0       0   
KASSEL          0         0         1       0           0         0       0   
LJUBLJANA       0         0         6       0           0         0       0   
MAASTRICHT      0         0         0       0           0         0       0   
MADRID          0         1        14      10          16         0       0   
MUNCHENB   